### Downloading PPS precipitation data and converting to parquet files 

This notebook takes PPS precipitation data and makes it into a parquet file (not a csv because csv uses more memory). 
Just need to change the folder path, then run the code, and then save the combined_df as a parquet file using the next line of code

In [1]:
import os
import h5py
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
from tqdm import tqdm
from datetime import datetime, timezone
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
folder_path = "/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/2024/03"

dataframes =[]

#midwest lat/lon
lat_min, lat_max = 37, 49
lon_min, lon_max = -104, -80

#list of hdf5 files in folder
hdf5_files = [file for file in os.listdir(folder_path) if file.endswith('.HDF5')]

#looping through each file + tqdm 
for file in tqdm(hdf5_files, desc="Processing HDF5 Files", unit="file"):
    file_path = os.path.join(folder_path, file)

    with h5py.File(file_path, 'r') as hdf:
        grid_group = hdf['Grid']

        precipitation_data = grid_group['precipitation'][0, :, :] #need to remove first dimension
        latitudes = grid_group['lat'][:] #1D array
        longitudes = grid_group['lon'][:] #1D array
        times = grid_group['time'][0] #single timestamp for all in one file 

        randomError = grid_group['randomError'][0, :, :]
        probabilityLiquidPrecipitation = grid_group['probabilityLiquidPrecipitation'][0, :, :]
        precipitationQualityIndex = grid_group['precipitationQualityIndex'][0, :, :]

        #make lat/lon 2D grid 
        lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

        #need to flatten lat/lon grid for filtering
        lat_flat = lat_grid.ravel()
        lon_flat = lon_grid.ravel()
        precipitation_flat = precipitation_data.ravel()
        randomError_flat = randomError.ravel()
        probabilityLiquidPrecipitation_flat = probabilityLiquidPrecipitation.ravel()
        precipitationQualityIndex_flat = precipitationQualityIndex.ravel()

        #create mask for midwest region 
        mask = (lat_flat >= lat_min) & (lat_flat <= lat_max) & (lon_flat >= lon_min) & (lon_flat <= lon_max)

        #apply mask 
        lat_flat = lat_flat[mask]
        lon_flat = lon_flat[mask]
        precipitation_flat = precipitation_flat[mask]
        randomError_flat = randomError_flat[mask]
        probabilityLiquidPrecipitation_flat = probabilityLiquidPrecipitation_flat[mask]
        precipitationQualityIndex_flat = precipitationQualityIndex_flat[mask]

        #want datetime
        epoch = datetime(1980, 1, 6, 0, 0, 0)
        time_value = pd.to_datetime(times, unit="s", origin=epoch)

        #creating df
        df = pd.DataFrame({
            "latitude": lat_flat,
            "longitude": lon_flat,
            "precipitation": precipitation_flat,
            "randomError": randomError_flat,
            "probabilityLiquidPrecipitation": probabilityLiquidPrecipitation_flat,
            "precipitationQualityIndex": precipitationQualityIndex_flat,
            "date": time_value.date(),
            "time_": time_value.time(),
        })

        #only want + precip values
        df = df[df["precipitation"] >= 0]

        dataframes.append(df)

#combinging dfs 
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    print("Combined DataFrame shape:", combined_df.shape)
else:
    print("no valid data found in the files")



Processing HDF5 Files: 100%|█| 1488/1488 [23:34<00:


Combined DataFrame shape: (42854400, 8)


In [3]:
combined_df.to_parquet("/Volumes/share-sat2/SAT/TEMPO/Toireasa/pps_HH_mar_precip.parquet",engine = 'pyarrow', index=False)

In [4]:
df = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/pps_HH_mar_precip.parquet',engine = 'pyarrow')

In [5]:
df

,latitude,longitude,precipitation,randomError,probabilityLiquidPrecipitation,precipitationQualityIndex,date,time_
0,37.049999,-103.949997,0.0,0.24,100,0.818,2024-03-14,11:00:00
1,37.049999,-103.849998,0.0,0.24,100,0.818,2024-03-14,11:00:00
2,37.049999,-103.750000,0.0,0.24,100,0.818,2024-03-14,11:00:00
3,37.049999,-103.649994,0.0,0.24,100,0.818,2024-03-14,11:00:00
4,37.049999,-103.549995,0.0,0.24,100,0.818,2024-03-14,11:00:00
...,...,...,...,...,...,...,...,...
42854395,48.950001,-80.449997,0.0,0.24,100,0.640,2024-03-12,04:00:00
42854396,48.950001,-80.349998,0.0,0.24,100,0.640,2024-03-12,04:00:00
42854397,48.950001,-80.250000,0.0,0.24,100,0.640,2024-03-12,04:00:00
42854398,48.950001,-80.150002,0.0,0.24,100,0.640,2024-03-12,04:00:00


In [7]:
unique_dates = df['date'].unique()
unique_dates

array([datetime.date(2024, 3, 14), datetime.date(2024, 3, 21),
       datetime.date(2024, 3, 11), datetime.date(2024, 3, 29),
       datetime.date(2024, 3, 3), datetime.date(2024, 3, 10),
       datetime.date(2024, 3, 26), datetime.date(2024, 3, 27),
       datetime.date(2024, 3, 25), datetime.date(2024, 3, 2),
       datetime.date(2024, 3, 8), datetime.date(2024, 3, 6),
       datetime.date(2024, 3, 12), datetime.date(2024, 3, 20),
       datetime.date(2024, 3, 31), datetime.date(2024, 3, 7),
       datetime.date(2024, 3, 1), datetime.date(2024, 3, 18),
       datetime.date(2024, 3, 23), datetime.date(2024, 3, 5),
       datetime.date(2024, 3, 16), datetime.date(2024, 3, 9),
       datetime.date(2024, 3, 30), datetime.date(2024, 3, 4),
       datetime.date(2024, 3, 13), datetime.date(2024, 3, 22),
       datetime.date(2024, 3, 24), datetime.date(2024, 3, 17),
       datetime.date(2024, 3, 28), datetime.date(2024, 3, 19),
       datetime.date(2024, 3, 15)], dtype=object)